# Numpy 를 통한 효율적인 비트 연산

* 기본적으로 Numpy는 C를 통하여 벡터 연산을 시행하기때문에 python 내에서 for문을 통한 바이너리 데이터 처리보다 수십배 빠르게 처리가 가능합니다

In [2]:
# 백문이불여일견. 한번 봅시다

import numpy as np
from timeit import Timer

li = list(range(10**6))
li_np = np.array(li)

def python_mod():
    return [num % (2**16 - 1) for num in li]

def numpy_mod():
    return li_np % (2**16 - 1)

print(f'파이썬 소요시간: {min(Timer(python_mod).repeat(10, 10))}')
print(f'넘파이 소요시간: {min(Timer(numpy_mod).repeat(10, 10))}')

파이썬 소요시간: 0.18920254199474584
넘파이 소요시간: 0.011115082990727387


--------
### 진행하기에 앞서 효율적인 파일 입출력 코드를 알아봅시다
* numpy 처리 시, 일반적인 open보다 np.fromfile (byte 변수로 이미 있으면 np.frombuffer) 나 np.memmap 으로 불러옵니다

In [ ]:
import shutil

#np.memmap은 읽어온 파일에 그대로 overwrite를 하기 때문에 shutil을 이용하여 빠르게 복사를 해줍니다

shutil.copyfile('old_file.bin','data.bin')

data = np.memmap('data.bin', dtype = np.uint8, mode = 'r+')

def main():
    '''
    대충 작업들...
    :return:
    '''
    pass

data.flush() # flush를 이용하여 디스크에 write!

-----------
### Command for Bitwise-operate Using Numpy

```python
import numpy as np
np.bitwise_and(x1, x2) # likely &
# 이뿐만이 아니라 np.bitwise_or,xor,not 도 지원.
np.right_shift(x, n) # likely << 2
np.left_shift(x, n) # likely >> 2

np.unpackbits(data) # unpacking to bits from bytes
np.packbits(data) # packing to bytes from bits

arr = np.array([], dtype = np.uint8) # assign arr as numpy type

arr.astype('uint8') # data type redifine

```

----------
### 실제 binary data를 가지고 처리를 해봅시다

In [106]:
import numpy as np

data = b"Hello, World! " * 40000 # width = 14bytes

with open('data.bin', 'wb') as f:
    f.write(data)

# size = 520KB

In [114]:
data = np.frombuffer(data[0:14*5], dtype=np.uint8).copy() # 혹시모를 참조이슈 및 오류들을 예방하기 위해 카피로 합시다
data = data.reshape(-1, 14) # data width는 14bytes로 reshape 함수로 행 -1 (열에 맞춰서 행 자동편성) 열 14을 만들어줌

In [115]:
np.bitwise_or(data, 0xff) # np.bitwise_ 시리즈로 비트연산을 할수있음. 다만 reshape와 달리 자체적으로 변수값을 변경하지 않음

array([[255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
        255],
       [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
        255],
       [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
        255],
       [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
        255],
       [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255,
        255]], dtype=uint8)

In [116]:
np.bitwise_right_shift(data, 3) # data << 3 연산과 동일함. np.right_shift와 동일한 함수이니 bitwise를 빼고 사용해도 된다.

array([[ 9, 12, 13, 13, 13,  5,  4, 10, 13, 14, 13, 12,  4,  4],
       [ 9, 12, 13, 13, 13,  5,  4, 10, 13, 14, 13, 12,  4,  4],
       [ 9, 12, 13, 13, 13,  5,  4, 10, 13, 14, 13, 12,  4,  4],
       [ 9, 12, 13, 13, 13,  5,  4, 10, 13, 14, 13, 12,  4,  4],
       [ 9, 12, 13, 13, 13,  5,  4, 10, 13, 14, 13, 12,  4,  4]],
      dtype=uint8)

In [117]:
int(np.sum(data & 0b1))
# vector 연산을 통해서 for문 없이 한번에 비트연산까지 가능함.
# 예시는 data 내 홀수 원소들의 개수를 알아내는 함수.

25

In [118]:
data.tobytes()

b'Hello, World! Hello, World! Hello, World! Hello, World! Hello, World! '

In [119]:
data

array([[ 72, 101, 108, 108, 111,  44,  32,  87, 111, 114, 108, 100,  33,
         32],
       [ 72, 101, 108, 108, 111,  44,  32,  87, 111, 114, 108, 100,  33,
         32],
       [ 72, 101, 108, 108, 111,  44,  32,  87, 111, 114, 108, 100,  33,
         32],
       [ 72, 101, 108, 108, 111,  44,  32,  87, 111, 114, 108, 100,  33,
         32],
       [ 72, 101, 108, 108, 111,  44,  32,  87, 111, 114, 108, 100,  33,
         32]], dtype=uint8)

In [120]:
data[:,7] = 119 # ASCII 'w'

In [ ]:
data.tobytes() # 모든 줄의 'W' 가 'w'로 바뀐것을 알수있음!

--------

In [147]:
import os
data = os.urandom(100000000) + (b'Sync!' + os.urandom(30000)) * 50 + b'NNNN'
sync = b'Sync!'

def find_sync_indices(data: bytes, sync: bytes) -> np.ndarray:
    """
    data 안에서 sync 바이트 시퀀스가 시작되는 모든 인덱스를 반환
    """
    data_arr = np.frombuffer(data, dtype=np.uint8)
    sync_arr = np.frombuffer(sync, dtype=np.uint8)

    n = len(sync_arr)
    if n == 0 or len(data_arr) < n:
        return np.array([], dtype=int)

    # 슬라이딩 윈도우 비교
    matches = np.all(
        data_arr[np.arange(n)[:, None] + np.arange(len(data_arr) - n + 1)]
        == sync_arr[:, None],
        axis=0
    )

    return np.where(matches)[0]

result = find_sync_indices(data, sync)

# 100mb data 기준으로 search가 1.5~2초밖에 안걸린다!

### 슬라이딩 윈도우 알고리즘이 뭘까요?!

In [195]:
a = np.array([1,2,3,5,7,2,3,3,3,1,2,3])
b = np.array([1,2])

# a행렬에서 [1,2]의 인덱스 위치를 슬라이딩 윈도우를 통하여 찾아봅시다

n = len(b)

windows = a[np.arange(n)[:, None] + np.arange(len(a) - n + 1)]
# windows = [[1 2 3 5 7 2 3 3 3 1 2]        [[1]   <- b[:, None]
#            [2 3 5 7 2 3 3 3 1 2 3]]        [2]]  를 각 열마다 비교하며 트루값 반환!

In [197]:
print(windows == b[:, None])

[[ True False False False False False False False False  True False]
 [ True False False False  True False False False False  True False]]


In [199]:
matches = np.all(windows == b[:, None], axis=0)

In [201]:
matches

array([ True, False, False, False, False, False, False, False, False,
        True, False])

In [207]:
np.where(matches)[0]

array([0, 9])